In [1]:
#!pip install redis

In [2]:
import hashlib

def get_image_key(file_path: str) -> str:
    """
    Gera chave única para cache de imagem via hash SHA256 do path.

    Args:
        file_path: Path completo da imagem (ex: /mnt/HD/image.jpg)

    Returns:
        Chave Redis no formato 'img:hash'
    """
    return f"img:{hashlib.sha256(file_path.encode()).hexdigest()}"

In [3]:
print(get_image_key("/mnt/HD/image.jpg"))

img:90e65bb7cb6d5cdebc49e2686e3306e5ded5bcc6a980ff559e0a31167f5950e9


In [4]:
print(get_image_key("/mnt/HD/image.jpg"))

img:90e65bb7cb6d5cdebc49e2686e3306e5ded5bcc6a980ff559e0a31167f5950e9


In [5]:

import redis
import hashlib
import json
import io
import cv2
import numpy as np
from typing import Optional, Dict, Any

# Cliente Redis singleton (ajuste host/port conforme ambiente)
redis_client = redis.Redis(
    host='192.168.18.253',  # Use 'redis' se rodar dentro do docker-compose
    port=6380,  # Porta externa; use 6379 se interno ao compose
    password='rdt_cache_pass',
    db=0,
    decode_responses=False  # Mantém bytes para imagens
)



def listar_chaves(pattern="*"):
    """
    Lista todas as chaves do Redis usando SCAN para evitar travar o servidor.
    pattern: pode ser "*", "img:*", "cache:*" etc.
    """
    cursor = 0
    chaves = []

    while True:
        cursor, batch = redis_client.scan(cursor=cursor, match=pattern, count=500)
        chaves.extend(batch)
        if cursor == 0:
            break

    # Como decode_responses=False, as chaves vêm em bytes → converte para str
    chaves = [k.decode("utf-8") for k in chaves]

    return chaves


file_txt = open('cache_img2.txt', 'w')

# Exemplo de uso
chaves = listar_chaves()
print(f"Total de chaves encontradas: {len(chaves)}")
# for k in chaves:
#     print(k)
#     if k.startswith('img:'):
#         file_txt.write(f"{k}\n")
    
# file_txt.close()


Total de chaves encontradas: 1280


In [6]:
def get_cache_stats() -> Dict[str, Any]:
    """
    Retorna estatísticas do Redis (uso de memória, total de chaves).

    Returns:
        Dict com 'used_memory', 'maxmemory', 'total_keys'
    """
    info = redis_client.info('memory')
    total_keys = redis_client.dbsize()
    return {
        'used_memory': info.get('used_memory_human', 'N/A'),
        'maxmemory': info.get('maxmemory_human', 'N/A'),
        'total_keys': total_keys
    }



def list_keys(pattern: str = "*", decode: bool = True, limit: Optional[int] = None):
    """
    Lista chaves do Redis usando SCAN, de forma segura.

    Args:
        pattern: padrão para filtro (ex: '*', 'img:*', 'api:gps_predict:*')
        decode: se True, decodifica bytes para str (utf-8)
        limit: se informado, limita o número de chaves retornadas

    Returns:
        Lista de chaves (str ou bytes)
    """
    cursor = 0
    keys_collected = []

    while True:
        cursor, keys = redis_client.scan(cursor=cursor, match=pattern, count=500)
        keys_collected.extend(keys)

        if limit is not None and len(keys_collected) >= limit:
            keys_collected = keys_collected[:limit]
            break

        if cursor == 0:
            break

    if decode and not redis_client.connection_pool.connection_kwargs.get("decode_responses", False):
        keys_collected = [k.decode("utf-8") for k in keys_collected]

    return keys_collected


def get_key_ttl(key: str) -> int:
    """
    Retorna o TTL (em segundos) de uma chave específica.

    Args:
        key: nome da chave (ex: 'img:abcdef...')

    Returns:
        TTL em segundos:
            -1 se a chave existe mas não tem expiração
            -2 se a chave não existe
             n >= 0 se tem expiração
    """
    # Se o cliente não decodifica, precisamos garantir que a key seja str/bytes coerente
    ttl = redis_client.ttl(key)
    return ttl


def debug_cache_snapshot(prefix: str = "*", max_keys: int = 50) -> Dict[str, Any]:
    """
    Retorna um snapshot simples do cache para inspeção rápida.

    Args:
        prefix: padrão de chave (ex: 'img:*', 'api:*', '*')
        max_keys: máximo de chaves a listar

    Returns:
        Dict com:
            'pattern': padrão usado
            'total_keys_db': total de chaves no DB
            'sampled_keys': lista de chaves (até max_keys)
            'stats': resultado de get_cache_stats()
    """
    keys = list_keys(pattern=prefix, limit=max_keys)
    stats = get_cache_stats()
    return {
        "pattern": prefix,
        "total_keys_db": stats["total_keys"],
        "sampled_keys": keys,
        "stats": stats,
    }


In [7]:
if __name__ == "__main__":
    # Ver todas as chaves de imagem
    img_keys = list_keys("/mnt*")
    print(f"Total de img:* = {len(img_keys)}")
    for k in img_keys[:10]:
        print(" -", k, "TTL:", get_key_ttl(k))

    # Snapshot geral
    from pprint import pprint
    pprint(debug_cache_snapshot("/mnt*", max_keys=20))


Total de img:* = 1112
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004585_cam0.jpg TTL: 86392
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004523_cam3.jpg TTL: 86281
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004550_cam1.jpg TTL: 86284
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004639_cam1.jpg TTL: 86298
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004479_cam3.jpg TTL: 86277
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004864_cam1.jpg TTL: 86326
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004879_cam3.jpg TTL: 86328
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004456_cam1.jpg TTL: 86274
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004736_cam1.jpg TTL: 86309
 - /mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004593_cam3.jpg TTL: 86291
{'pattern': '/mnt*',
 'sampled_keys': ['/mnt/hd1/Extracoes/PGRS_2025/Cube/subindoserra_Cube_004585_cam0.jpg',
                  '/mnt/hd1/Extracoes/PGRS_2025/